# This notebook is used to test other pseudonymization tools

Tools to compare with `mailcom`:
- [Presidio](https://github.com/microsoft/presidio/)
- [Scrubadub](https://github.com/LeapBeyond/scrubadub)

## Install tools if needed

Python 3.10

In [ ]:
%pip install scrubadub

In [ ]:
%pip install scrubadub_spacy scrubadub_stanford

In [ ]:
%pip install "presidio_analyzer[transformers]"
%pip install presidio_anonymizer
# python -m spacy download en_core_web_sm

## Try out the tools

### Presidio

In [ ]:
from presidio_analyzer import AnalyzerEngine
from presidio_analyzer.nlp_engine import TransformersNlpEngine
from presidio_anonymizer import AnonymizerEngine
from presidio_analyzer.nlp_engine import NlpEngineProvider

In [ ]:
text = """
Dear Dr. Emma Müller,

I hope this email finds you well.

I'm writing to you today from NextAI AG in Munich, Germany. We're keen to discuss the exciting developments from the recent Heidelberg AI Summit 2025. Specifically, we were very interested in the presentation on "Next-Generation Robotics" that took place in Room 306 of the main convention center.

Our team at NextAI AG would love to set up a quick call to discuss potential collaborations following the insights shared. We're thinking of a brief chat around July 15th.

Please let us know if July 15th works for you, or suggest an alternative time.

Best regards,
Max Schneider

--- Forwarded message ---
From: info@heidelbergaisummit.de
Date: Friday, 27 June 2025 at 14:30:06
Subject: Heidelberg AI Summit 2025 - Thank You!

Dear Attendees,

Thank you for making the Heidelberg AI Summit 2025 a resounding success! We truly appreciate your participation and engagement. We look forward to seeing you at future events.

Sincerely, 
The Heidelberg AI Summit Team

"""

In [ ]:
# run with custom model config file

config_content = """
nlp_engine_name: transformers
models:
    - lang_code: en
      model_name:
        spacy: en_core_web_sm
        transformers: xlm-roberta-large-finetuned-conll03-english
ner_model_configuration:
labels_to_ignore:
- O
model_to_presidio_entity_mapping:
    PER: PERSON
    LOC: LOCATION
    ORG: ORGANIZATION
    MISC: MISC
low_confidence_score_multiplier: 0.4
low_score_entity_names: []
"""

# save config to a yaml file
with open("config.yaml", "w") as f:
    f.write(config_content)

# Create NLP engine based on configuration file
provider = NlpEngineProvider(conf_file="config.yaml")
nlp_engine = provider.create_engine()

# Set up the engine, loads the NLP module (spaCy model by default) 
# and other PII recognizers
analyzer = AnalyzerEngine(nlp_engine=nlp_engine)

# Call analyzer to get results
results = analyzer.analyze(text=text, language='en')
print(results)

# Analyzer results are passed to the AnonymizerEngine for anonymization

anonymizer = AnonymizerEngine()

anonymized_text = anonymizer.anonymize(text=text, analyzer_results=results)

print(anonymized_text)

In [ ]:
# run with transformers model name

# Define which transformers model to use
model_config = [{"lang_code": "en", "model_name": {
    "spacy": "en_core_web_sm",  # use a small spaCy model for lemmas, tokens etc.
    "transformers": "xlm-roberta-large-finetuned-conll03-english"
    }
}]

nlp_engine = TransformersNlpEngine(models=model_config)

# Set up the engine, loads the NLP module (spaCy model by default) 
# and other PII recognizers
analyzer = AnalyzerEngine(nlp_engine=nlp_engine)

# Call analyzer to get results
results = analyzer.analyze(text=text, language='en')
print(results)

# Analyzer results are passed to the AnonymizerEngine for anonymization

anonymizer = AnonymizerEngine()

anonymized_text = anonymizer.anonymize(text=text, analyzer_results=results)

print(anonymized_text)

### Scrubadub

In [ ]:
import scrubadub, scrubadub_spacy, scrubadub_stanford # scrubadub_address requires additional setup, see https://scrubadub.readthedocs.io/en/stable/addresses.html

In [ ]:
text = """
Dear Dr. Emma Müller,

I hope this email finds you well.

I'm writing to you today from NextAI AG in Munich, Germany. We're keen to discuss the exciting developments from the recent Heidelberg AI Summit 2025. Specifically, we were very interested in the presentation on "Next-Generation Robotics" that took place in Room 306 of the main convention center.

Our team at NextAI AG would love to set up a quick call to discuss potential collaborations following the insights shared. We're thinking of a brief chat around July 15th.

Please let us know if July 15th works for you, or suggest an alternative time.

Best regards,
Max Schneider

--- Forwarded message ---
From: info@heidelbergaisummit.de
Date: Friday, 27 June 2025 at 14:30:06
Subject: Heidelberg AI Summit 2025 - Thank You!

Dear Attendees,

Thank you for making the Heidelberg AI Summit 2025 a resounding success! We truly appreciate your participation and engagement. We look forward to seeing you at future events.

Sincerely, 
The Heidelberg AI Summit Team

"""

In [ ]:
# add external detectors
scrubber = scrubadub.Scrubber()

# only use one at a time, otherwise the pseudonymized text will have multiple tags for the same entity
scrubber.add_detector(scrubadub_spacy.detectors.SpacyEntityDetector)
# scrubber.add_detector(scrubadub_spacy.detectors.SpacyNameDetector)

# adding the below detectors make the process non-stop running, don't know why
# scrubber.add_detector(scrubadub_stanford.detectors.StanfordEntityDetector)

In [ ]:
pseudonymized_text = scrubber.clean(text)
print(pseudonymized_text)